<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/vietnam_go.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# @title Run command
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""

    Starting environment setup...
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            escaped_line = line.replace("'", "\'").strip()
            display(HTML(f""""""), display_id='scroll_update')
        process.wait()

# 2. Consolidated commands for Makro, Scrapling, and OCR (Updated for Colab Support)
commands_to_run = [
    "apt-get update",
    "apt-get install -y google-chrome-stable chromium-browser chromium-chromedriver",
    "pip install selenium bs4 beautifulsoup4 pandas polars xlsxwriter fastexcel curl_cffi scrapling playwright patchright msgspec browserforge nest_asyncio itables chromedriver-autoinstaller webdriver-manager google-colab-selenium easyocr pillow requests -q",
    "patchright install chromium",
    "patchright install-deps",
    "playwright install chromium",  # <-- Add this
    "playwright install-deps"       # <-- Add this
]

run_and_scroll(commands_to_run)
print("\n✅ Environment ready! Let's go.")


✅ Environment ready for Makro! Let's go.


In [6]:
from playwright.async_api import async_playwright
import asyncio
import json

async def find_api():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"
        )

        # Capture all API requests
        api_calls = []
        page.on("response", lambda resp: api_calls.append({
            "url": resp.url,
            "status": resp.status,
            "type": resp.request.resource_type
        }))

        await page.goto(
            "https://sieuthi-go.vn/categories/home-care-i.67",
            wait_until="networkidle", timeout=30000
        )
        await asyncio.sleep(3)

        # Dismiss store popup
        try:
            store = await page.wait_for_selector("text=ĐANG CHỌN", timeout=5000)
            if store:
                await store.click()
                await asyncio.sleep(3)
        except:
            pass

        # --- Filter API/XHR calls ---
        print("🔎 API calls found:\n")
        for call in api_calls:
            if call["type"] in ["xhr", "fetch"] and call["status"] == 200:
                url = call["url"]
                # Skip tracking/analytics
                if any(x in url for x in ["google", "facebook", "analytics",
                                           "pixel", "tracking", "fonts",
                                           "gtm", "cdn", ".js", ".css"]):
                    continue
                print(f"   📡 {url[:150]}")

        # --- Try to fetch product API responses ---
        print("\n\n🔎 Looking for product data in API responses...\n")
        for call in api_calls:
            if call["type"] in ["xhr", "fetch"] and call["status"] == 200:
                url = call["url"]
                if any(x in url.lower() for x in ["product", "category", "item",
                                                    "catalog", "api"]):
                    print(f"🎯 Potential product API: {url[:200]}")

                    # Try to get the response body
                    try:
                        # Re-fetch the API
                        resp = await page.request.get(url)
                        body = await resp.json()

                        # Print a sample
                        sample = json.dumps(body, ensure_ascii=False, indent=2)[:2000]
                        print(f"   📦 Response sample:\n{sample}\n")
                    except:
                        print(f"   ⚠️ Could not parse response\n")

        await browser.close()

await find_api()

🔎 API calls found:

   📡 https://sieuthi-go.vn/api/init
   📡 https://wa.onelink.me/v1/onelink
   📡 https://sieuthi-go.vn/api/app-init
   📡 https://sieuthi-go.vn/api/store-get-list-store?lang=vi&platform=2
   📡 https://wa.appsflyersdk.com/events?site-id=M7ZR87BU2QFZUh7huS5oK5
   📡 https://sieuthi-go.vn/api/save-store
   📡 https://sieuthi-go.vn/api/loyalty-cart-save?storeId=21
   📡 https://wa.onelink.me/v1/onelink?af_id=4248dea9-86eb-406a-b163-f50eeb9ca478-p
   📡 https://sieuthi-go.vn/api/getSetting
   📡 https://sieuthi-go.vn/api/loyalty-cart-view?storeId=21&platform=2&lang=vi
   📡 https://sieuthi-go.vn/api/list-campaign?platform=2&lang=vi
   📡 https://sieuthi-go.vn/api/get-categories?store=102&lang=vi&platform=2
   📡 https://sieuthi-go.vn/api/list-campaign?platform=2&lang=vi
   📡 https://sieuthi-go.vn/api/list-category-quick-ecommerce
   📡 https://sieuthi-go.vn/api/order2_listProduct?platform=2&lang=vi


🔎 Looking for product data in API responses...

🎯 Potential product API: https://si

In [7]:
from playwright.async_api import async_playwright
from deep_translator import GoogleTranslator
import polars as pl
import asyncio
import time

BASE_URL = "https://sieuthi-go.vn/categories/home-care-i.67"
all_data = []
seen_names = set()

async def scrape():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"
        )

        print(f"\n📄 Loading initial page: {BASE_URL}")
        await page.goto(BASE_URL, wait_until="networkidle", timeout=30000)
        await asyncio.sleep(3)

        # Dismiss store popup ONCE
        try:
            store = await page.wait_for_selector("text=ĐANG CHỌN", timeout=5000)
            if store:
                await store.click()
                await asyncio.sleep(3)
                print("   🏪 Store selected")
        except:
            pass

        page_num = 1

        while page_num <= 20:
            print(f"\n📄 Scraping batch {page_num}...")

            # SMART WAIT: Pause until products are actually present in the DOM
            try:
                await page.wait_for_selector(".stick_camp_prod_item", state="visible", timeout=15000)
            except Exception:
                print("   ⚠️ Timed out waiting for products to load. Stopping.")
                break

            await asyncio.sleep(2)

            # Scrape products
            cards = await page.query_selector_all(".stick_camp_prod_item")
            if not cards:
                print("   ⚠️ No products found. Stopping.")
                break

            before = len(all_data)
            for card in cards:
                name_el = await card.query_selector("p.chakra-text.css-138rv0j")
                name = (await name_el.inner_text()).strip() if name_el else None

                if not name or name in seen_names:
                    continue
                seen_names.add(name)

                sale_el = await card.query_selector("b.chakra-text.css-aqfyqq")
                sale = None
                if sale_el:
                    s = (await sale_el.inner_text()).strip()
                    sale = s.replace("đ", "").replace(",", "").replace(".", "").strip()

                orig_el = await card.query_selector("del.chakra-text.css-nxr2hb")
                original = None
                if orig_el:
                    o = (await orig_el.inner_text()).strip()
                    original = o.replace("đ", "").replace(",", "").replace(".", "").strip()

                disc_el = await card.query_selector("span.p-1")
                discount = (await disc_el.inner_text()).strip() if disc_el else None

                all_data.append({
                    "product_name_vi": name,
                    "sale_price": int(sale) if sale else None,
                    "original_price": int(original) if original else None,
                    "discount_pct": discount,
                })

            print(f"   ✅ +{len(all_data) - before} new (total: {len(all_data)})")

            # Click "See more products" using the corrected class selector
            btn = await page.query_selector("button.ant-btn-round.ant-btn-variant-outlined")
            if not btn or await btn.is_disabled():
                print("🏁 No more 'See more products' button. Done!")
                break

            await btn.scroll_into_view_if_needed()
            await asyncio.sleep(1)

            try:
                # Force click via JavaScript
                await btn.evaluate("node => node.click()")
                print("   🖱️ Clicked 'See more products' (via JavaScript)")

                await asyncio.sleep(5)
                page_num += 1

                # Safeguard
                if len(all_data) == before:
                    print("   ⚠️ Click executed, but no new products appeared. Ending loop.")
                    break
            except Exception as e:
                print(f"🏁 Failed to click 'See more products': {e}. Done!")
                break

        await browser.close()

await scrape()

# --- Result Processing ---
# 1. Initialize Polars DataFrame
df = pl.DataFrame(all_data)
print(f"\n🎯 Scraped: {len(df)} unique products")

if len(df) > 0:
    print("\n🌐 Translating to English...")
    translator = GoogleTranslator(source="vi", target="en")

    def translate_batch(names, batch_size=20):
        translated = []
        for i in range(0, len(names), batch_size):
            batch = names[i:i+batch_size]
            joined = " ||| ".join(batch)
            try:
                result = translator.translate(joined)
                parts = [p.strip() for p in result.split("|||")]
                if len(parts) == len(batch):
                    translated.extend(parts)
                else:
                    for n in batch:
                        try: translated.append(translator.translate(n))
                        except: translated.append(n)
            except:
                for n in batch:
                    try: translated.append(translator.translate(n))
                    except: translated.append(n)

            print(f"   ✅ {min(i+batch_size, len(names))}/{len(names)}")

        return translated

    # 2. Extract series to a list for translation and add it back using with_columns
    translated_names = translate_batch(df["product_name_vi"].to_list())
    df = df.with_columns(
        pl.Series(name="product_name_en", values=translated_names)
    )

    # 3. Select and reorder columns
    df = df.select([
        "product_name_en",
        "product_name_vi",
        "sale_price",
        "original_price",
        "discount_pct"
    ])

    print(f"\n🎯 Final: {len(df)} products")
    print(df.select(["product_name_en", "sale_price", "original_price"]).head(10))

    # 4. Save to CSV (No index=False needed in Polars)
    # Using utf8-bom if you intend to open the file directly in Excel to preserve Vietnamese characters
    with open("go_vietnam_home_care.csv", "wb") as f:
        f.write(b"\xef\xbb\xbf") # Write UTF-8 BOM manually for Excel compatibility
        f.write(df.write_csv().encode("utf-8"))

    print("\n💾 Saved to go_vietnam_home_care.csv")


📄 Loading initial page: https://sieuthi-go.vn/categories/home-care-i.67
   🏪 Store selected

📄 Scraping batch 1...
   ✅ +55 new (total: 55)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 2...
   ✅ +15 new (total: 70)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 3...
   ✅ +15 new (total: 85)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 4...
   ✅ +15 new (total: 100)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 5...
   ✅ +15 new (total: 115)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 6...
   ✅ +11 new (total: 126)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 7...
   ✅ +10 new (total: 136)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 8...
   ✅ +13 new (total: 149)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 9...
   ✅ +9 new (total: 158)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping

To Do:
Next Load watchlist

In [8]:
# @title UDF - transform
def parse_product_name(df: pl.DataFrame, name_col: str) -> pl.DataFrame:
    """
    Extracts Volume, Unit, and product_id from a specified product name column.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.
        name_col (str): The column name containing the product string.

    Returns:
        pl.DataFrame: A DataFrame with three new columns: 'Volume', 'Unit', and 'product_id'.
    """

    # Regex for Volume and Unit:
    # Group 1 extracts numbers (including decimals).
    # Group 2 extracts common units. (Placed longest words first to prevent partial matches like 'l' in 'liter').
    vol_unit_pattern = r"(?i)(\d+(?:\.\d+)?)\s*(liters?|kg|g|ml|l)\b"

    # Regex for Product ID:
    # Extracts trailing digits that come after a hyphen.
    id_pattern = r"-\s*(\d+)\s*$"

    return df.with_columns(
        # Extract volume and cast to a Float
        pl.col(name_col)
            .str.extract(vol_unit_pattern, 1)
            .cast(pl.Float64)
            .alias("Volume"),

        # Extract unit and standardize it to lowercase
        pl.col(name_col)
            .str.extract(vol_unit_pattern, 2)
            .str.to_lowercase()
            .alias("Unit"),

        # Extract the product ID
        pl.col(name_col)
            .str.extract(id_pattern, 1)
            .alias("product_id")
    )

In [9]:
# --- Example Usage ---
parsed_df = parse_product_name(df, "product_name_en")

In [10]:
parsed_df

product_name_en,product_name_vi,sale_price,original_price,discount_pct,Volume,Unit,product_id
str,str,i64,i64,str,f64,str,str
"""OMO Matic laundry detergent fo…","""Nước giặt OMO Matic quần áo bé…",244000,null,null,3.8,"""kg""","""97461"""
"""OMO Matic front-load laundry d…","""Nước giặt OMO Matic cửa trước …",204000,null,null,3.8,"""kg""","""97423"""
"""Comfort fabric softener, 1-tim…","""Nước xả vải Comfort 1 lần …",261000,null,null,3.7,"""liter""","""50398"""
"""Comfort fabric softener, gentl…","""Nước xả vải Comfort dịu nhẹ th…",267000,null,null,3.6,"""liter""","""88155"""
"""Comfort fabric softener with l…","""Nước xả vải Comfort hương nước…",276000,null,null,3.7,"""l""","""98642"""
…,…,…,…,…,…,…,…
"""OMO Comfort laundry detergent …","""Bột giặt OMO Comfort tinh dầu …",142500,null,null,2.6,"""kg""","""50008"""
"""Aba washing powder 770g - 1712…","""Bột giặt Aba 770g - 17122""",35900,null,null,770.0,"""g""","""17122"""
"""Ariel quick clean laundry dete…","""Bột giặt Ariel sạch nhanh 4Kg …",199500,null,null,4.0,"""kg""","""54070"""


Exp

In [13]:
# @title Run Multiple Stores
async def scrape_all_stores():
    # A dictionary mapping store_id to store_name
    target_stores = {
        "3": "GO! LONG BIÊN",
        "16": "GO! ĐÀ NẴNG",
        "21": "GO! AN LẠC"
    }

    for s_id, s_name in target_stores.items():
        print(f"\n{'='*40}")
        print(f"🚀 STARTING SCRAPE FOR: {s_name}")
        print(f"{'='*40}")

        # Call your cookie experiment function
        await scrape_experiment_cookie(store_id=s_id, store_name=s_name)

        # Be polite to the server between store switches
        await asyncio.sleep(5)

# Run the loop
await scrape_all_stores()


🚀 STARTING SCRAPE FOR: GO! LONG BIÊN
🍪 Injecting cookie for Store ID: 3 (GO! LONG BIÊN)

📄 Loading page: https://sieuthi-go.vn/categories/home-care-i.67

📄 Scraping batch 1...
   ✅ Found 62 products.
      📦 Added: [3] Nước giặt OMO Matic quần áo bé...
      📦 Added: [3] Nước giặt OMO Matic cửa trước ...
      📦 Added: [3] Nước xả vải Comfort 1 lần ...
      📦 Added: [3] Nước xả vải Comfort dịu nhẹ th...
      📦 Added: [3] Nước xả vải Comfort hương nước...

📊 DataFrame Preview:
shape: (5, 4)
┌──────────┬───────────────┬─────────────────────────────────┬────────────┐
│ store_id ┆ store_name    ┆ product_name_vi                 ┆ sale_price │
│ ---      ┆ ---           ┆ ---                             ┆ ---        │
│ str      ┆ str           ┆ str                             ┆ i64        │
╞══════════╪═══════════════╪═════════════════════════════════╪════════════╡
│ 3        ┆ GO! LONG BIÊN ┆ Nước giặt OMO Matic quần áo bé… ┆ 244000     │
│ 3        ┆ GO! LONG BIÊN ┆ Nước giặt OMO